# Combining — concat, merge, jointures

Ce notebook travaille directement sur les **4 tables réelles** du dataset accidents.
C'est la structure qui justifie le choix de ce dataset :
un accident réel est décrit par 4 tables liées par des clés, exactement comme en production.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from _data import (
    load_accidents_caracteristiques,
    load_accidents_lieux,
    load_accidents_vehicules,
    load_accidents_usagers,
)

caract   = load_accidents_caracteristiques()
lieux    = load_accidents_lieux()
vehicules = load_accidents_vehicules()
usagers  = load_accidents_usagers()

print(f"CARACTERISTIQUES : {caract.shape}")
print(f"LIEUX            : {lieux.shape}")
print(f"VEHICULES        : {vehicules.shape}")
print(f"USAGERS          : {usagers.shape}")

## 1. Pourquoi plusieurs tables ?

```
CARACTERISTIQUES         LIEUX
(1 ligne / accident)     (1 ligne / accident)
┌──────────┬──────┬───┐  ┌──────────┬──────┬─────┐
│ Num_Acc  │ dep  │...│  │ Num_Acc  │ catr │ vma │
└──────────┴──────┴───┘  └──────────┴──────┴─────┘
     │                        │
     └────────── Num_Acc ──────┘
          │
     VEHICULES                USAGERS
     (1 ligne / véhicule)     (1 ligne / usager)
     ┌──────────┬─────────┬───┐  ┌──────────┬─────────┬──────┐
     │ Num_Acc  │ num_veh │...│  │ Num_Acc  │ num_veh │ grav │
     └──────────┴─────────┴───┘  └──────────┴─────────┴──────┘
          │                            │
          └─── Num_Acc + num_veh ───────┘
```

Un accident peut impliquer plusieurs véhicules, chacun avec plusieurs usagers.
Stocker tout dans une seule table créerait une explosion combinatoire
et des redondances massives — d'où les 4 tables normalisées.

In [ ]:
# Vérification des clés
print("Accidents uniques (CARACTERISTIQUES) :", caract["Num_Acc"].nunique())
print("Accidents uniques (LIEUX)            :", lieux["Num_Acc"].nunique())
print("Accidents uniques (VEHICULES)        :", vehicules["Num_Acc"].nunique())
print("Accidents uniques (USAGERS)          :", usagers["Num_Acc"].nunique())
print()
print("Véhicules par accident (médiane) :",
      vehicules.groupby("Num_Acc")["num_veh"].count().median())
print("Usagers par accident  (médiane) :",
      usagers.groupby("Num_Acc")["id_usager"].count().median())

## 2. `concat` — empiler des DataFrames

In [ ]:
# Empilage vertical (axis=0) : ajouter des lignes
# Cas typique : réunir deux exports mensuels ou deux années
premier_semestre = caract.query("mois <= 6")
second_semestre  = caract.query("mois > 6")

annee_entiere = pd.concat([premier_semestre, second_semestre], ignore_index=True)
print(f"{len(premier_semestre)} + {len(second_semestre)} = {len(annee_entiere)} lignes")

In [ ]:
# ignore_index=True : repart d'un index 0, 1, 2… propre
# Sans ignore_index, l'index original est conservé — peut créer des doublons d'index
sans_reset = pd.concat([premier_semestre, second_semestre])
avec_reset = pd.concat([premier_semestre, second_semestre], ignore_index=True)

print("Index dupliqués sans ignore_index :", sans_reset.index.duplicated().sum())
print("Index dupliqués avec ignore_index :", avec_reset.index.duplicated().sum())

In [ ]:
# Empilage horizontal (axis=1) : ajouter des colonnes
# Cas typique : deux DataFrames alignées sur le même index
# Attention : aligne sur l'index — si les index ne matchent pas, NaN
caract_small  = caract[["Num_Acc", "dep", "mois"]].head(5).set_index("Num_Acc")
lieux_small   = lieux[["Num_Acc", "catr", "vma"]].head(5).set_index("Num_Acc")

pd.concat([caract_small, lieux_small], axis=1)

## 3. `merge` — inner join

L'inner join ne conserve que les lignes qui trouvent une correspondance dans les deux tables.

In [ ]:
# CARACTERISTIQUES + LIEUX → une ligne par accident avec toutes les infos
accidents = caract.merge(lieux, on="Num_Acc", how="inner")
print(f"CARACTERISTIQUES : {len(caract)} lignes")
print(f"LIEUX            : {len(lieux)} lignes")
print(f"Après inner join : {len(accidents)} lignes")

In [ ]:
# Les accidents perdus (dans CARACTERISTIQUES mais pas dans LIEUX)
perdus = len(caract) - len(accidents)
print(f"Accidents sans correspondance LIEUX : {perdus} ({perdus/len(caract)*100:.1f}%)")

## 4. Left join — conserver toutes les lignes gauches

In [ ]:
# Pour chaque usager, récupérer les infos de l'accident
# left = USAGERS (toutes les lignes conservées)
# right = CARACTERISTIQUES (infos de l'accident)
usagers_enrichis = usagers.merge(caract, on="Num_Acc", how="left")

print(f"USAGERS          : {len(usagers)} lignes")
print(f"Après left join  : {len(usagers_enrichis)} lignes  (même nombre — aucun usager perdu)")
print(f"NaN dans 'dep'   : {usagers_enrichis['dep'].isna().sum()} (usagers sans accident dans CARACTERISTIQUES)")

In [ ]:
# Illustration : répartition de la gravité par type de collision
# grav : 1=indemne, 2=tué, 3=blessé hospitalisé, 4=blessé léger
# col  : type de collision (dans CARACTERISTIQUES)
(
    usagers_enrichis
    .query("grav == 2")           # uniquement les tués
    .groupby("col")["id_usager"]
    .count()
    .rename("nb_tues")
    .sort_values(ascending=False)
)

## 5. Outer join et `indicator` — diagnostiquer les manquants

L'outer join conserve toutes les lignes des deux tables.
Le paramètre `indicator=True` ajoute une colonne `_merge` qui indique
d'où vient chaque ligne — indispensable pour diagnostiquer la qualité d'une jointure.

In [ ]:
diagnostic = caract[["Num_Acc"]].merge(
    lieux[["Num_Acc"]],
    on="Num_Acc",
    how="outer",
    indicator=True,
)

diagnostic["_merge"].value_counts()

In [ ]:
# Accidents présents dans LIEUX mais absents de CARACTERISTIQUES
orphelins = diagnostic.query("_merge == 'right_only'")
print(f"{len(orphelins)} accidents dans LIEUX sans correspondance dans CARACTERISTIQUES")

## 6. Pipeline complet — question métier sur 3 tables

**Question** : pour chaque type de collision (`col`), quelle est la proportion d'usagers tués ?

In [ ]:
# Dictionnaire de labels pour col
LABELS_COLLISION = {
    1: "Deux veh. frontale",
    2: "Deux veh. arrière",
    3: "Deux veh. côté",
    4: "En chaîne",
    5: "Multiples",
    6: "Autre",
    7: "Sans collision",
}

(
    usagers                                                    # 1 ligne par usager
    .merge(caract[["Num_Acc", "col"]], on="Num_Acc", how="left")  # ajouter type de collision
    .assign(tue=lambda df_: (df_["grav"] == 2).astype(int))    # indicateur tué
    .groupby("col")
    .agg(
        nb_usagers=("id_usager", "count"),
        nb_tues   =("tue",       "sum"),
    )
    .assign(
        pct_tues=lambda df_: (df_["nb_tues"] / df_["nb_usagers"] * 100).round(2),
        type_collision=lambda df_: df_.index.map(LABELS_COLLISION),
    )
    .sort_values("pct_tues", ascending=False)
    .reset_index(drop=True)
    [["type_collision", "nb_usagers", "nb_tues", "pct_tues"]]
)

## 7. Merge sur plusieurs clés — VEHICULES + USAGERS

VEHICULES et USAGERS partagent deux clés : `Num_Acc` ET `num_veh`.
Il faut les deux pour identifier un véhicule précis dans un accident.

In [ ]:
# Pour chaque usager, récupérer le type de véhicule dans lequel il était
usagers_vehicules = usagers.merge(
    vehicules[["Num_Acc", "num_veh", "catv", "motor"]],
    on=["Num_Acc", "num_veh"],
    how="left",
)
print(f"Usagers          : {len(usagers)}")
print(f"Après join       : {len(usagers_vehicules)}")
usagers_vehicules.head(3)

In [ ]:
# Tués par motorisation du véhicule
# motor : 1=thermique, 2=hybride, 3=électrique, 4=hydrogène, 5=autre
(
    usagers_vehicules
    .assign(tue=lambda df_: (df_["grav"] == 2).astype(int))
    .groupby("motor")
    .agg(
        nb_usagers=("id_usager", "count"),
        nb_tues   =("tue",       "sum"),
    )
    .assign(pct_tues=lambda df_: (df_["nb_tues"] / df_["nb_usagers"] * 100).round(2))
    .sort_values("pct_tues", ascending=False)
)

## Récapitulatif

| Opération | Méthode | Résultat |
|---|---|---|
| Empiler des lignes | `pd.concat([df1, df2])` | Plus de lignes |
| Empiler des colonnes | `pd.concat([df1, df2], axis=1)` | Plus de colonnes |
| Lignes en commun seulement | `.merge(df2, on="clé", how="inner")` | Intersection |
| Toutes les lignes gauches | `.merge(df2, on="clé", how="left")` | Toutes les lignes de df1 |
| Toutes les lignes des deux | `.merge(df2, on="clé", how="outer")` | Union |
| Diagnostiquer les manquants | `.merge(..., indicator=True)` | Colonne `_merge` |
| Jointure sur plusieurs clés | `.merge(df2, on=["k1", "k2"])` | Clé composite |